In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans



class WeightedSet:

    def __init__(self, P, W, Y=None):
        # P is of size nXd
        # W is of size 1Xn
        # Y is of size nXk
        ##todo add checker

        self.P = np.array(P)
        if (P.ndim == 1):
            self.P = self.P.reshape(-1, 1)
        self.n = self.P.shape[0]
        self.d = self.P.shape[1]
        self.Y = Y
        self.W = np.array(W).reshape(1, -1)
        self.dtype = P.dtype

        if (self.Y is not None) and (P.shape[0] != Y.shape[0]):
            self.Y = self.Y.reshape(self.n, -1)

        self.sum_W = np.sum(self.W);  # print (self.W.shape)
        self.weighted_sum = self.P.T.dot(self.W.T)

    # p must be a 1 dimensional nparray (p.shape = (d,) )
    # w is a number
    def add_point(self, p, w, y):
        self.P = np.append(self.P, [p], axis=0)
        self.W = np.append(self.W, w)
        if not self.Y is None: self.Y = np.append(self.Y, y)
        self.n = self.n + 1
        self.sum_W = self.sum_W + w
        self.weighted_sum = self.weighted_sum + w * p
        
# Load the Iris dataset
data = load_iris()
X = data.data
W = np.ones(X.shape[0])  # Assign uniform weights
Pset = WeightedSet(X, W, X.shape[1] + 1)

def calculate_sensitivities(X, n_clusters=3):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(X)
    centers = kmeans.cluster_centers_
    labels = kmeans.labels_
    sensitivities = np.zeros(X.shape[0])
    
    for idx, point in enumerate(X):
        cluster = labels[idx]
        distance = np.linalg.norm(point - centers[cluster])
        sensitivities[idx] = distance**2
    
    return sensitivities

def sample_coreset(X, sensitivities, coreset_size=50):
    probabilities = sensitivities / np.sum(sensitivities)
    indices = np.random.choice(len(X), size=coreset_size, replace=False, p=probabilities)
    coreset = X[indices]
    return coreset, indices


# Calculate sensitivities and sample coreset
sensitivities = calculate_sensitivities(X)
coreset_size = 50  # You can vary this size as needed
coreset, coreset_indices = sample_coreset(X, sensitivities, coreset_size=coreset_size)
print(len(coreset_indices))

# Perform K-Means clustering on the coreset
kmeans_coreset = KMeans(n_clusters=3, random_state=42)
kmeans_coreset.fit(coreset)

# perform k-means on the full dataset
kmeans_full = KMeans(n_clusters=3, random_state=42)
kmeans_full.fit(X)


def calculate_wcss(X, labels, centers):
    wcss = 0
    for idx, point in enumerate(X):
        cluster = labels[idx]
        wcss += np.linalg.norm(point - centers[cluster])**2
    return wcss

# Calculate WCSS for coreset and full dataset
wcss_coreset = calculate_wcss(coreset, kmeans_coreset.labels_, kmeans_coreset.cluster_centers_)
wcss_full = calculate_wcss(X, kmeans_full.labels_, kmeans_full.cluster_centers_)

print(f"WCSS on Coreset: {wcss_coreset}")
print(f"WCSS on Full Dataset: {wcss_full}")


150
WCSS on Coreset: 37.21265325670499
WCSS on Full Dataset: 78.85566582597727
